In [13]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langgraph.checkpoint.memory import InMemorySaver
from langchain_anthropic import ChatAnthropic
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import operator
import time
from time import sleep
from langchain_core.messages import SystemMessage, HumanMessage

In [14]:
#State
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str
    step3: str


In [15]:
#Define steps

def step1(state: CrashState) -> CrashState:
    print("Step 1 executed")
    
    return {'step 1': "done", 'input': state['input']}

def step2(state: CrashState) -> CrashState:
    print("Step 2 hanging... now manually interrpurt from notebook toolbar (STOP BUTTON)")
    time.sleep(30)
    return {'step 2': "done"}

def step3(state: CrashState) -> CrashState:
    print("Step 3 executed")
    
    return {'done': True}


In [16]:
#Build graph

graph = StateGraph(CrashState)

#add nodes
graph.add_node("step1",step1)
graph.add_node("step2",step2)
graph.add_node("step3",step3)

#add edges
graph.add_edge(START, "step1")
graph.add_edge("step1", "step2")        
graph.add_edge("step2", "step3")
graph.add_edge("step3", END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("Running graph: Please manually interrupt duting step 2...")
    
    workflow.invoke({"input": "start"}, config={"configurable": {"thread_id": "1"}})
    # workflow.invoke(None, config={"configurable": {"thread_id": "1"}})
    
    
except KeyboardInterrupt:
    print("Kernel manually iterrupted. (crash simulated)")

Running graph: Please manually interrupt duting step 2...
Step 1 executed
Step 2 hanging... now manually interrpurt from notebook toolbar (STOP BUTTON)
